# QickworkspaceV2 — Direct Hardware Measurements

Use this notebook for daily experiment development: edit Python dictionaries and native QICK programs, then acquire one experiment at a time.
Frequencies are in **MHz**, times in **us**, phases in **degrees**, and gains in normalized QICK units.
Every `lab.run()` acquires from the connected board and saves the raw IQ, compiled configuration, and analysis results.

Check the wiring profile and initial device values selected by `lab/project.yaml` before connecting.
Calibrate readout and GE pulses before proceeding to EF, gate, and two-qubit experiments. Run the relevant cells individually rather than using Run All.
Adjust the example values for your device. Hardware readout-frequency sweeps in this notebook require direct, tProc-controlled readout. For MUX or PYNQ-controlled readout-frequency sweeps, use `notebooks/advanced_measurements.ipynb`.

Use `config_all["Q1"].update(res_gain_ge=0.15, res_sigma=0.01)` to edit working settings in one call.
`qb.for_run(steps=101, ...)` creates an independent configuration for one run; `config_all.update_all(...)` updates all qubits.
Run configurations support normal dictionary updates, including keyword arguments and iterable key/value pairs.
T1 GE and EF are independently maintained in `QickworkspaceV2/experiments/t1_ge.py` and `t1_ef.py`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "lab/project.yaml").exists():
    ROOT = ROOT.parent
if not (ROOT / "lab/project.yaml").exists():
    raise FileNotFoundError("Open this notebook from the project root directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from qick.asm_v2 import QickSweep1D
from QickworkspaceV2 import Measurement, BaseProgram, ExperimentConfig
from QickworkspaceV2.plotting import LivePlot
from lab.config import CONNECTION, DATA_PATH, make_config

In [ ]:
config_all = make_config()
qubit = "Q1"
PY_AVG = 5

# Edit daily settings here, or update make_config() in lab/config.py.
qb = config_all[qubit]  # Reusable working settings
qb.update(res_gain_ge=0.15, res_sigma=0.01)
config_all.update_all(relax_delay=200)  # Explicitly update every qubit
config_all[qubit].for_run()

## Connect

This cell connects to the board without starting a measurement sequence. Edit the IP address, port, and data directory here.
`res_sigma` affects `res_pulse_type="arb"` or `"flat_top"`. A `"const"` pulse has no envelope.

In [ ]:
CONNECTION = {**CONNECTION, "ns_host": "192.168.10.82", "ns_port": 8888, "proxy_name": "myqick"}
DATA_PATH = ROOT / "data"
lab = Measurement.from_pyro4(**CONNECTION, data_path=DATA_PATH)
soc, soccfg = lab.soc, lab.soccfg
print(soccfg)

## Time of Flight

Acquire decimated I/Q with `reps=1`. `PY_AVG` controls software averaging.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.time_of_flight import TimeOfFlightProgram
Program = TimeOfFlightProgram
run_cfg = config_all[qubit].for_run()
run_cfg.update(reps=1, res_length=0.5, ro_length=3.0, trig_time=0.0, relax_delay=100)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

In [ ]:
result.plot(signal="real")

## Resonator Spectroscopy

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.resonator_spec import ResonatorSpecProgram
Program = ResonatorSpecProgram
qb = config_all[qubit]
STEPS = 101
START_FREQ = qb["res_freq_ge"] - 5
STOP_FREQ = qb["res_freq_ge"] + 5
run_cfg = qb.for_run(
    steps=STEPS,
    res_freq_ge=QickSweep1D("freqloop", START_FREQ, STOP_FREQ),
    res_gain_ge=0.15,
    relax_delay=0,
)
# Further edits can use run_cfg.update(...) or run_cfg["res_gain_ge"] = 0.12.

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

In [ ]:
# Run this cell to update working settings after reviewing the fit and plot.
if result.is_good():
    config_all[qubit].update(res_freq_ge=result.metrics[qubit]["center"])

## Qubit Spectroscopy GE

Sweep the drive frequency for two-tone spectroscopy.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.qubit_spec_ge import QubitSpecGEProgram
Program = QubitSpecGEProgram
run_cfg = config_all[qubit].for_run()
STEPS = 101
START_FREQ = run_cfg["qb_freq_ge"] - 20
STOP_FREQ = run_cfg["qb_freq_ge"] + 20
run_cfg.update(steps=STEPS, qb_freq_ge=QickSweep1D("freqloop", START_FREQ, STOP_FREQ),
               qb_gain_ge=0.05, qb_flat_top_length_ge=2.0, relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

In [ ]:
if result.is_good():
    config_all[qubit].update(qb_freq_ge=result.metrics[qubit]["center"])

## Qubit Spectroscopy EF

Prepare the e state with a calibrated GE pi pulse before probing the EF transition.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.qubit_spec_ef import QubitSpecEFProgram
Program = QubitSpecEFProgram
run_cfg = config_all[qubit].for_run()
STEPS = 101
START_FREQ = run_cfg["qb_freq_ef"] - 20
STOP_FREQ = run_cfg["qb_freq_ef"] + 20
run_cfg.update(steps=STEPS, qb_freq_ef=QickSweep1D("freqloop", START_FREQ, STOP_FREQ),
               qb_gain_ef=0.05, qb_flat_top_length_ef=2.0, relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

In [ ]:
if result.is_good():
    config_all[qubit].update(qb_freq_ef=result.metrics[qubit]["center"])

## Power Rabi GE

Edit the pulse settings and gain sweep below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.power_rabi_ge import PowerRabiGEProgram
Program = PowerRabiGEProgram
qb = config_all[qubit]
# Keep this waveform for later gates so the fitted pi gain remains applicable.
qb.update(pulse_type_ge="arb", sigma_ge=0.02)
run_cfg = qb.for_run(steps=101, qb_gain_ge=QickSweep1D("gainloop", 0.01, 0.45), relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

In [ ]:
if result.is_good():
    config_all[qubit].update(pi_gain_ge=result.metrics[qubit]["pi_gain"])
    config_all[qubit].update(pi2_gain_ge=result.metrics[qubit]["pi2_gain"])

## Time Rabi GE

Sweep the duration of a constant pulse. Keep this calibration separate from pi-gain calibration for a Gaussian envelope.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.time_rabi_ge import TimeRabiGEProgram
Program = TimeRabiGEProgram
run_cfg = config_all[qubit].for_run()
run_cfg.update(steps=101, qb_length_ge=QickSweep1D("lengthloop", 0.02, 2.0),
               qb_gain_ge=0.15, relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

## T1 GE

`wait_us` sets the free-evolution delay. The fitted `tau` is in us.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.t1_ge import T1GEProgram
Program = T1GEProgram
run_cfg = config_all[qubit].for_run(
    steps=81,
    wait_us=QickSweep1D("waitloop", 0, 100),
    relax_delay=200,
)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

## Ramsey GE

`detuning_mhz` (MHz) controls the phase ramp of the second pi/2 pulse. The fit does not infer the sign of the detuning or automatically update the qubit frequency.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.ramsey_ge import RamseyGEProgram
Program = RamseyGEProgram
run_cfg = config_all[qubit].for_run()
run_cfg.update(steps=101, wait_us=QickSweep1D("waitloop", 0, 30), detuning_mhz=0.2, relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

## Spin Echo GE

`wait_us` specifies the sum of both free-evolution intervals. Saved coordinates and plots use the actual compiled total waiting time.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.spin_echo_ge import SpinEchoGEProgram
Program = SpinEchoGEProgram
run_cfg = config_all[qubit].for_run()
run_cfg.update(steps=81, wait_us=QickSweep1D("waitloop", 0, 100), relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

## Power Rabi EF

Edit the pulse settings and gain sweep below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.power_rabi_ef import PowerRabiEFProgram
Program = PowerRabiEFProgram
qb = config_all[qubit]
# Keep this waveform for later gates so the fitted pi gain remains applicable.
qb.update(pulse_type_ef="arb", sigma_ef=0.02)
run_cfg = qb.for_run(steps=101, qb_gain_ef=QickSweep1D("gainloop", 0.01, 0.45), relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

In [ ]:
if result.is_good():
    config_all[qubit].update(pi_gain_ef=result.metrics[qubit]["pi_gain"])
    config_all[qubit].update(pi2_gain_ef=result.metrics[qubit]["pi2_gain"])

## Time Rabi EF

Sweep the duration of a constant pulse. Keep this calibration separate from pi-gain calibration for a Gaussian envelope.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.time_rabi_ef import TimeRabiEFProgram
Program = TimeRabiEFProgram
run_cfg = config_all[qubit].for_run()
run_cfg.update(steps=101, qb_length_ef=QickSweep1D("lengthloop", 0.02, 2.0),
               qb_gain_ef=0.15, relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

## T1 EF

`wait_us` sets the free-evolution delay. The fitted `tau` is in us.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.t1_ef import T1EFProgram
Program = T1EFProgram
run_cfg = config_all[qubit].for_run(
    steps=81,
    wait_us=QickSweep1D("waitloop", 0, 100),
    relax_delay=200,
)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

## Ramsey EF

`detuning_mhz` (MHz) controls the phase ramp of the second pi/2 pulse. The fit does not infer the sign of the detuning or automatically update the qubit frequency.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.ramsey_ef import RamseyEFProgram
Program = RamseyEFProgram
run_cfg = config_all[qubit].for_run()
run_cfg.update(steps=101, wait_us=QickSweep1D("waitloop", 0, 30), detuning_mhz=0.2, relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

## Spin Echo EF

`wait_us` specifies the sum of both free-evolution intervals. Saved coordinates and plots use the actual compiled total waiting time.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.spin_echo_ef import SpinEchoEFProgram
Program = SpinEchoEFProgram
run_cfg = config_all[qubit].for_run()
run_cfg.update(steps=81, wait_us=QickSweep1D("waitloop", 0, 100), relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

## Single Shot

Each repetition contains paired g/e readouts. Shots from every software-averaging round are retained.

Edit `run_cfg` below, then execute the acquisition cell.

In [ ]:
from QickworkspaceV2.experiments.single_shot import SingleShotProgram
Program = SingleShotProgram
run_cfg = config_all[qubit].for_run()
run_cfg.update(reps=2000, reset_wait_us=200, relax_delay=200)

In [ ]:
result = lab.run(Program, run_cfg, py_avg=PY_AVG, on_progress=LivePlot())
print(result.metrics)
print(result.path)

In [ ]:
print(result[qubit].shots.shape)
if result.is_good():
    config_all[qubit].update(ro_phase=result.metrics[qubit]["rotation_deg"])
    config_all[qubit].update(ro_threshold=result.metrics[qubit]["threshold"])
# After calibrating IQ rotation, use config_all.update_all(iq_process="real") for later runs.

## Multi-Qubit Gate-Level T1

Each target keeps its own frequency, gain, and ADC mapping. `config_all.for_run("Q1", "Q2", ...)` supports multiple targets; add Q3 to the selection after configuring its hardware and device entries.
`qb.x()` and `qb.halfx()` use the calibrated pulses for that qubit. Place simultaneous operations inside `parallel()`.

In [ ]:
class GateT1Program(BaseProgram):
    def _initialize(self, cfg):
        self.setup_device(cfg, gates=True)
        self.add_sweep_loop(cfg, cfg["wait_us"])

    def _body(self, cfg):
        with self.parallel():
            for qb in self.qubits.values():
                qb.x()
        self.delay_auto(cfg["wait_us"], tag="evolution")
        self.measure(cfg)

In [ ]:
Program = GateT1Program
run_cfg = config_all.for_run("Q1", "Q2")
run_cfg["qubits"]["Q2"]["res_gain_ge"] = 0.10
run_cfg.update(steps=81, wait_us=QickSweep1D("waitloop", 0, 100), relax_delay=200)

In [ ]:
from QickworkspaceV2.experiments.t1_ge import analyze as fit_t1
result = lab.run(Program, run_cfg, py_avg=PY_AVG, analyze=fit_t1, on_progress=LivePlot())
print(result.metrics)
result.plot()

## Write a Native Pulse-Level Program

Edit `_initialize` and `_body` directly in the notebook. Add experiment-specific parameters to `run_cfg`; no experiment registration or parameter-schema changes are required.

In [ ]:
class QubitSpecProgram(BaseProgram):
    """QICK program for two-tone qubit spectroscopy."""
    def _initialize(self, cfg):
        self.setup_resonator(cfg)
        self.setup_qubit_gen(cfg, "ge")
        self.add_loop("freqloop", cfg["steps"])
        self.setup_qb_pulse(cfg, "ge", name="qb_pulse", pulse_type="flat_top")

    def _body(self, cfg):
        self.send_readoutconfig(ch=cfg["ro_ch"], name="myro", t=0)
        if cfg.get("cooling", False):
            self.apply_cool(cfg)
            self.cooling_body(cfg)
        self.pulse(ch=cfg["qb_ch"], name="qb_pulse", t=0)
        self.delay_auto(cfg["readout_wait"])
        self.measure(cfg)

In [ ]:
Program = QubitSpecProgram
run_cfg = config_all[qubit].for_run()
center = run_cfg["qb_freq_ge"]
run_cfg.update(steps=101, qb_freq_ge=QickSweep1D("freqloop", center-20, center+20),
               qb_gain_ge=0.05, qb_flat_top_length_ge=2.0, readout_wait=0.05, relax_delay=200)

In [ ]:
from QickworkspaceV2.experiments.qubit_spec_ge import analyze as fit_spec
result = lab.run(Program, run_cfg, py_avg=PY_AVG, analyze=fit_spec)
result.plot()

The native QICK API is also available. This example bypasses the measurement manager and does not save data automatically:

```python
prog = QubitSpecProgram(soccfg, reps=run_cfg["reps"], final_delay=run_cfg["relax_delay"], cfg=run_cfg)
iq = prog.acquire(soc, rounds=PY_AVG, progress=True)
```

To inspect compiled instructions without starting acquisition:

```python
prog = lab.compile(Program, run_cfg)
print(prog)
```

## Save Working Settings and Reload Results

Every managed acquisition is already saved. `acquisition.h5` contains the original acquisition; analysis revisions are stored separately.
Execute the next cell when you want to save the current `config_all` working settings. It does not overwrite the original wiring profile.

In [ ]:
config_all.save(ROOT / "lab/working_config.yaml")
# Next session: config_all = ExperimentConfig.load(ROOT / "lab/working_config.yaml")

In [ ]:
loaded = lab.load(result.run_id)
print(loaded.path)
loaded.plot(signal="abs", residuals=True)

See `notebooks/advanced_measurements.ipynb` for additional independent experiments, MUX frequency host sweeps, coupler measurements, and the NVIDIA worker.